# Clustering visualization

## 0. 준비

In [ ]:
import os
import platform
import glob
import random

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from gensim.models import KeyedVectors

### 폴더 경로 지정

In [ ]:
os_system = platform.system() # 맥북은 Darwin, 윈도우는 Windows

# 현재 프로젝트 폴더 위치 지정. os.getcwd()는 지금 코드 실행하는 현 위치를 출력해줍니다.
study1_dir = os.getcwd()

# model path
model_path = '\\..\\pretrained\\GoogleNews-vectors-negative300.bin' if os_system == 'Windows' else '/../pretrained/GoogleNews-vectors-negative300.bin'

# data/processed 폴더 위치 지정
processed_data_dir = study1_dir + ('\\data\\processed\\' if os_system == 'Windows' else '/data/processed/')

# graph 이미지 저장할 폴더 위치 지정
graph_image_dir = study1_dir + ('\\graph\\clustering\\' if os_system == 'Windows' else '/graph/clustering/')
# 폴더 없으면 생성
os.makedirs(graph_image_dir, exist_ok=True)

# 2차원벡터 파일 위치 지정
fitted_vectors_dir = study1_dir + ('\\loaded_vectors\\*.npy' if os_system == 'Windows' else '/loaded_vectors/*.npy')


### word2vec 모델 로딩

In [ ]:
word2vec_model = KeyedVectors.load_word2vec_format(study1_dir + model_path, binary=True)

### 데이터 로딩

In [ ]:
tbl_data = pd.read_csv(processed_data_dir + 'data_after_preprocessing.csv', encoding='ISO-8859-1')
tbl_data.columns

### 단어 데이터로 벡터 만들기

In [ ]:
seed_words = ['key', 'money', 'friend']
target_words = ['money', 'friend']
n_respond_words = 30 # 하나의 시드당 30개의 단어 응답
n_subject = len(tbl_data) # 210
n_dim_of_vector = 300

# 벡터구하기
for seed_word in seed_words: # key, money, friend
    word_columns = [seed_word + str(i) for i in range(1, n_respond_words+1)] #  key1~30, money1~30, friend1~30

    for column in word_columns:
        # vector field 생성
        tbl_data[column + '_vec'] = np.empty(n_subject, dtype=object)

        # 피험자 한 명의 응답 단어들 벡터 처리
        for i_subject in range(n_subject):
            try:
                response_word = tbl_data.iloc[i_subject][column]
                if pd.isna(response_word) or len(response_word.strip()) == 0:# NaN, 값이 빈 칸 & 응답안해서 '', ' '로 저장된 경우 걸러내기
                    tbl_data[column + '_vec'][i_subject] = None
                    continue

                if isinstance(response_word, str):
                    response_word = response_word.split()
                    response_word = [response_word for response_word in response_word if response_word not in ['is', 'a','to','of','and']]
                    if len(response_word) == 0: # 앞에서 걸러져서 결과가 없으면, 넘어가기
                        tbl_data[column + '_vec'][i_subject] = None
                        continue

                    vec_word2vec = np.zeros((n_dim_of_vector, 0))  # 300차원의 빈 행렬 생성

                    for i_el in range(len(response_word)):
                        try:
                            vec_word2vec_in = word2vec_model[response_word[i_el]]
                        except:
                            vec_word2vec_in = word2vec_model[response_word[i_el].capitalize()]
                        # reshape: 벡터의 형태를 바꿔줄뿐. 300을 600 or 90으로 바꿀 순 없다.
                        vec_word2vec_in = vec_word2vec_in.reshape((n_dim_of_vector, 1))
                        vec_word2vec = np.hstack((vec_word2vec, vec_word2vec_in))  # 수평으로 벡터 쌓기
                    # 각 열(단어 벡터)에 대한 평균 계산
                    average_vector = np.mean(vec_word2vec, axis=1)
                    tbl_data[column + '_vec'][i_subject] = average_vector #38 tear30
            except:
                pass

## 1. 시각화

### 데이터 준비

In [ ]:
vectors_data = tbl_data.iloc[:, 91:]

# 아예 응답안한 피험자 행 제거
all_columns_are_none = vectors_data.isna().all(axis=1)
vectors_data = vectors_data[~all_columns_are_none]
len(vectors_data)

In [ ]:
# 타겟 단어 벡터화하기
target_word_vec = []
for target_word in target_words:
    target_word_vec.append(np.array(word2vec_model[target_word]))
target_word_vec = np.array(target_word_vec)


### 시각화 함수

In [ ]:
# 색깔 랜덤으로 생성하는 함수
def random_hex_color():
    r = lambda: np.random.randint(0, 256)
    return "#{:02X}{:02X}{:02X}".format(r(), r(), r())

In [ ]:
def get_scatter_of_vectors(num_total_subject, total_columns, response_vectors, target_vectors, target_words, lim: int, save_path: str, is_save: bool):
    """
    num_total_subject: 총 피험자 수
    total_columns: 피험자별, 응답한 단어들의 모음. ex. { 0: [ 'tear1_vec', ... ], ...} 
    response_vectors: 피험자들이 응답한 단어들의 벡터 모음. 피험자 구분없이 차례로, 리스트에 2차원의 벡터들이 나열되어있다.
    target_vectors: 타겟 단어의 벡터 리스트. [(2,), (2,)]
    target_words: 타겟 단어의 str 리스트 ['money', 'friend']
    """
    plt.figure(figsize=(16, 16), dpi=300)

    total_words_count = 0
    for i_subject in range(num_total_subject):
        # if i_subject > 10:
        #     break
        # response words col
        valid_columns = total_columns[i_subject]
        num_response_words = len(valid_columns)

        x = []
        y = []

        for ind in range(total_words_count, (total_words_count + num_response_words)): # 이전까지 ~ 이전+현재
            x.append(response_vectors[ind][0])
            y.append(response_vectors[ind][1])
        
        # 피험자별 랜덤 색깔 추출해서 plot
        color = random_hex_color() 
        for i in range(len(x)):
            plt.scatter(x[i], y[i], c=color, s=300, alpha=1, edgecolors='white')
        
        total_words_count += num_response_words # 누적해서 카운트하기 위함

    # target
    target_x = []
    target_y = []

    for value in target_vectors:
        target_x.append(value[0])
        target_y.append(value[1])

    for i, word in enumerate(target_words):
        plt.scatter(target_x[i], target_x[i], marker="X", c='red', s=2000, alpha=1, edgecolors='white')
        plt.annotate(word,
                        xy=(target_x[i], target_x[i]),
                        xytext=(5, 2),
                        textcoords='offset points',
                        ha='right',
                        va='bottom')

    # x축과 y축 범위 조절
    # plt.xlim(-lim, lim)
    # plt.ylim(-lim, lim)
    plt.grid(True)
    # # plt.show()

    if is_save:
        plt.savefig(save_path, dpi=300)


### 전체 피험자

#### - 전체 피험자, 전체 응답단어(90개) 벡터, 컬럼 데이터 준비

In [ ]:
## tsne 모델에 넣기위해, 모든 피험자의 모든 응답(빈값빼고)들 + 타겟 단어 2개를 하나의 데이터로 묶어내는 작업

all_subjects_tokens = []
for i_subject in range(len(vectors_data)):
    vectors_of_subject = []
    for column, value in vectors_data.iloc[i_subject, :].items():
        if not (isinstance(value, (int, float)) and value != 0) and (value is not 0) and value is not None:
            vectors_of_subject.append((column, value)) 
    all_subjects_tokens.append(vectors_of_subject) # ex. subject_tokens[0]: 0번 피험자들의 벡터 정보들이 들어있음. [(컬럼이름, 벡터 ), ... ]
print(len(all_subjects_tokens), len(all_subjects_tokens[0])) # 210명 피험자, 0번 피험자는 89개 답함

# 응답 결과 전체에서 vector값만 뽑아내기. [ (300,), (300, ), ... ] 형태로 되어있음.
vectors = np.array([vec for subject_tokens in all_subjects_tokens for column, vec in subject_tokens])
combined_data = np.vstack((vectors, target_word_vec))
print(combined_data.shape)

# 피험자별로 응답 결과가 있는 컬럼값만 뽑아놓기
total_columns = {}
for i_subject, subject_tokens in enumerate(all_subjects_tokens):
    notnull_columns = []
    for column, vec in subject_tokens:
        notnull_columns.append(column)
    total_columns[i_subject] = notnull_columns

#### - tsne모델로 차원축소된 벡터 데이터들을 로딩해서 산점도에 그려내기

In [ ]:
def get_scatter_from_loaded_vectors(npy_file_path, save_topic, lim=500):
    # 저장된 TSNE 결과 불러오기
    loaded_fitted_vectors = np.load(npy_file_path)
    # 전체 데이터와 타겟 데이터의 결과 추출
    subject_new_vectors = loaded_fitted_vectors[:len(vectors)]
    target_new_vectors = loaded_fitted_vectors[len(vectors):]

    npy_file_name = os.path.basename(npy_file_path).split(".")[0]

    print(subject_new_vectors.shape, target_new_vectors.shape)
    get_scatter_of_vectors(num_total_subject = len(vectors_data),
                        total_columns = total_columns,
                        response_vectors = subject_new_vectors, 
                        target_vectors = target_new_vectors, 
                        target_words = target_words,
                        lim=lim,
                        save_path=f'{graph_image_dir}/{save_topic}_{npy_file_name}.png', 
                        is_save=True)

In [ ]:
npy_list = glob.glob(fitted_vectors_dir)
for npy in npy_list:
    get_scatter_from_loaded_vectors(npy_file_path = npy,
                                    save_topic='total_all_seeds')

### 전체 피험자의 seed별(30개)

#### - 피험자별 응답한 단어들 리스트 모으기

In [ ]:
valid_columns = {}
for i_subject in total_columns:
    key_columns = [column for column in total_columns[i_subject] if column.startswith('key')]
    money_columns = [column for column in total_columns[i_subject] if column.startswith('money')]
    friend_columns = [column for column in total_columns[i_subject] if column.startswith('friend')]
    valid_columns[i_subject] = {
        'key': key_columns,
        'money': money_columns,
        'friend': friend_columns
    }
len(valid_columns)

#### - seed별 벡터 모으기

In [ ]:
def get_vectors_of_seed_words(valid_columns, loaded_fitted_vectors):
    # seed별 벡터 모으기
    key_vectors = {}
    money_vectors = {}
    friend_vectors = {}
    total_cnt = 0
    for i_subject in range(len(valid_columns)): # 0~209
        num_key_columns = len(valid_columns[i_subject]['key'])
        num_money_columns = len(valid_columns[i_subject]['money'])
        num_friend_columns = len(valid_columns[i_subject]['friend'])
        # print('subject', i_subject, num_key_columns, num_money_columns, num_friend_columns)

        key_vectors[i_subject] = loaded_fitted_vectors[total_cnt:(total_cnt + num_key_columns)]
        money_vectors[i_subject] = loaded_fitted_vectors[(total_cnt + num_key_columns):(total_cnt + num_key_columns + num_money_columns)]
        friend_vectors[i_subject] = loaded_fitted_vectors[(total_cnt + num_key_columns + num_money_columns):(total_cnt + num_key_columns + num_money_columns + num_friend_columns)]

        total_num_of_subject = num_key_columns+num_money_columns+num_friend_columns
        total_cnt += total_num_of_subject

    # 타겟 단어 벡터 모으기
    target_vectors = loaded_fitted_vectors[-2:]

    return key_vectors, money_vectors, friend_vectors, target_vectors


#### - 차원 축소된 데이터로 산점도

In [ ]:
def get_scatter_of_seed_vectors(seed_vectors, valid_columns, target_vectors, target_words, lim: int, save_path: str, is_save: bool):
    """
    seed_vectors: 피험자별, 응답 단어의 벡터 리스트. {0: [(2,), (2,), ... ], ... }
    valid_columns: 피험자별, 응답한 단어들의 모음. ex. { 0: [ 'tear1_vec', ... ], ...} 
    target_vectors: 타겟 단어의 벡터 리스트. [(2,), (2,)]
    target_words: 타겟 단어의 리스트. ['money', 'friend']
    """
    plt.figure(figsize=(16, 16), dpi=300)

    for i_subject in range(len(valid_columns)): # 0~209
        x = []
        y = []
        for i, vec in enumerate(seed_vectors[i_subject]):
            x.append(vec[0])
            y.append(vec[1])
        # 피험자별 랜덤 색깔 추출해서 plot
        color = random_hex_color() 
        for i in range(len(x)):
            plt.scatter(x[i], y[i], c=color, s=50, alpha=1, edgecolors='white')
            
    # target
    target_x = []
    target_y = []
    for value in target_vectors:
        target_x.append(value[0])
        target_y.append(value[1])

    for i, word in enumerate(target_words):
        plt.scatter(target_x[i], target_x[i], marker="X", c='red', s=2000, alpha=1, edgecolors='white')
        plt.annotate(word,
                        xy=(target_x[i], target_x[i]),
                        xytext=(5, 2),
                        textcoords='offset points',
                        ha='right',
                        va='bottom')

    # x축과 y축 범위 조절
    # plt.xlim(-lim, lim)
    # plt.ylim(-lim, lim)
    plt.grid(True)
    # plt.show()

    if is_save:
        plt.savefig(save_path, dpi=300)


In [ ]:
npy_list = glob.glob(fitted_vectors_dir)

for i, npy in enumerate(npy_list):
    npy_file_name = os.path.basename(npy).split(".")[0]

    loaded_fitted_vectors = np.load(npy)
    key_vectors, money_vectors, friend_vectors, target_vectors = get_vectors_of_seed_words(valid_columns, loaded_fitted_vectors)
    # key
    get_scatter_of_seed_vectors(key_vectors, 
                                valid_columns,
                                target_vectors,
                                target_words,
                                lim = 500,
                                save_path = f'{graph_image_dir}/total_key_{npy_file_name}.png',
                                is_save = True)
    # money
    get_scatter_of_seed_vectors(money_vectors, 
                                valid_columns,
                                target_vectors,
                                target_words,
                                lim = 500,
                                save_path = f'{graph_image_dir}/total_money_{npy_file_name}.png',
                                is_save = True)
    # friend
    get_scatter_of_seed_vectors(friend_vectors, 
                                valid_columns,
                                target_vectors,
                                target_words,
                                lim = 500,
                                save_path = f'{graph_image_dir}/total_friend_{npy_file_name}.png',
                                is_save = True)